# Predicción de Ventas Semanales

Proyecto de regresión / series de tiempo listo para **Google Colab**.

**Pregunta:** ¿podemos predecir las ventas de la próxima semana?

**Target:** ventas

**Stack:** pandas, seaborn, scikit-learn, shap

Datos simulados de ventas semanales.

## 0. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
print("OK")

## 1. Generar serie de ventas semanales

In [ ]:
np.random.seed(RANDOM_STATE)
n = 200
week = np.arange(n)
trend = 2 * week
seasonal = 50 * np.sin(2 * np.pi * week / 52)
noise = np.random.normal(0, 15, n)
sales = 500 + trend + seasonal + noise
df = pd.DataFrame({
    "week": week,
    "sales": sales,
    "promo": np.random.choice([0, 1], n, p=[0.7, 0.3]),
    "holiday": np.random.choice([0, 1], n, p=[0.9, 0.1]),
})
# Features lag
df["sales_lag1"] = df["sales"].shift(1)
df["sales_lag4"] = df["sales"].shift(4)
df["ma4"] = df["sales"].rolling(4).mean()
df = df.dropna().reset_index(drop=True)
print(df.shape)
df.head()

## 2. EDA

In [ ]:
print(df.describe().T)

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(df["week"], df["sales"])
plt.title("Ventas semanales")
plt.xlabel("Semana")
plt.ylabel("Ventas")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=df, x="promo", y="sales", ax=axes[0])
axes[0].set_title("Ventas por promo")
sns.scatterplot(data=df, x="sales_lag1", y="sales", ax=axes[1])
axes[1].set_title("Lag-1 vs Ventas")
plt.tight_layout()
plt.show()

### Hallazgos
- Tendencia alcista + estacionalidad anual.
- Lags y promos influyen.
- Ideal para forecast de demanda.

## 3. Preparar datos

In [ ]:
X = df.drop(columns=["sales", "week"])
y = df["sales"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False, random_state=RANDOM_STATE
)
print(X_train.shape, X_test.shape)

## 4. Modelos

In [ ]:
modelos = {
    "Ridge": Ridge(random_state=RANDOM_STATE),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE),
    "Gradient Boosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
}

resultados = []
pipes = {}

for nombre, clf in modelos.items():
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model", clf)
    ])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    resultados.append({
        "modelo": nombre,
        "MAE": mean_absolute_error(y_test, y_pred),
        "RMSE": mean_squared_error(y_test, y_pred, squared=False),
        "R2": r2_score(y_test, y_pred),
    })
    pipes[nombre] = pipe

pd.DataFrame(resultados).sort_values("R2", ascending=False)

## 5. Feature importance

In [ ]:
rf = pipes["Random Forest"].named_steps["model"]
importancias = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(8, 4))
sns.barplot(x=importancias.values, y=importancias.index)
plt.title("Importancia — Random Forest")
plt.tight_layout()
plt.show()

## 6. SHAP

In [ ]:
!pip install -q shap

In [ ]:
import shap

rf_pipe = pipes["Random Forest"]
X_train_scaled = rf_pipe.named_steps["scaler"].transform(X_train)
model = rf_pipe.named_steps["model"]

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_train_scaled)

print("SHAP listo")

In [ ]:
shap.summary_plot(shap_values, X_train_scaled, feature_names=X.columns, max_display=6, show=False)
plt.title("SHAP Summary — Ventas")
plt.tight_layout()
plt.show()

## 7. Conclusiones

- Los lags y la media móvil dominan la predicción.
- Promos y feriados aportan señal.
- Se puede extender a datos reales de retail.

**Rango esperado (ejemplo):** R² > 0.80